# Job Finder — RAG Agent Development & Deployment

**ISA632 Group Project | Phase 2 of 2**

This notebook covers the second half of the RAG pipeline: enabling MLflow tracing, building and testing the LangChain agent, registering the model to Unity Catalog, and deploying it via the Databricks Agent Framework with a Review App.

**Run order:**
1. `01_DataPreparation.ipynb` — ingest documents, build Delta table, sync Vector Search index
2. `02_AgentDevelopment.ipynb` ← you are here

**Prerequisites:**
- `01_DataPreparation.ipynb` has been run and the Vector Search index `isa632_7474656346303369.boopatt.jobindex` is synced
- `agent.py` is present at `/Workspace/Users/boopatt@miamioh.edu/Project/agent.py`
- Catalog: `isa632_7474656346303369` | Schema: `boopatt`

## Step 1 — Install Dependencies

Install all required packages for the agent and MLflow integration. The kernel restarts automatically — re-run subsequent cells after the restart.

In [ ]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0

dbutils.library.restartPython()

## Step 2 — Enable MLflow Tracing

Enable LangChain auto-logging so that every agent invocation is automatically traced and logged in MLflow. This captures the full chain of retrieval → prompt → LLM response, making it easy to inspect and debug the pipeline.

The `VS_INDEX_NAME` environment variable is set here and read by `agent.py` at runtime to locate the correct Vector Search index.

In [ ]:
import os
import mlflow

mlflow.langchain.autolog()

os.environ["VS_INDEX_NAME"] = "isa632_7474656346303369.boopatt.jobindex"
print("MLflow tracing enabled. VS_INDEX_NAME set.")

## Step 3 — Load and Test the RAG Agent

Import the `AGENT` object from `agent.py`, which wraps the full LangChain pipeline:
- **LLM:** Databricks Llama 4 Maverick via `ChatDatabricks`
- **Retriever:** Databricks Vector Search tool
- **Agent type:** Tool-calling agent via LangChain `AgentExecutor`

Run a test query with `verbose=True` to inspect the reasoning steps — you should see the agent call the vector search tool before generating a final answer.

In [ ]:
import sys
import importlib

# Ensure latest version of agent.py is loaded
sys.path.insert(0, '/Workspace/Users/boopatt@miamioh.edu/Project')
if 'agent' in sys.modules:
    del sys.modules['agent']

from agent import AGENT

AGENT.agent.verbose = True

user_input = "what are the available roles for scala?"
request = {
    "input": [
        {"role": "user", "content": user_input}
    ]
}

resp = AGENT.predict(request)
print(resp)

## Step 4 — Register the Model to Unity Catalog

Log the agent as an MLflow model and register it to Unity Catalog. Registering the model:
- Creates a versioned, shareable artifact in the catalog
- Captures all dependencies (LangChain, Vector Search, MLflow versions) for reproducibility
- Declares the Vector Search index as a resource so it is automatically provisioned at serving time

The registered model name is `isa632_7474656346303369.boopatt.getstarted_job_listings`.

In [ ]:
import mlflow
from mlflow.models.resources import DatabricksVectorSearchIndex
from mlflow.models import infer_signature
from pkg_resources import get_distribution

mlflow.set_registry_uri("databricks-uc")

model_name = "isa632_7474656346303369.boopatt.getstarted_job_listings"
vs_index_table_name = "isa632_7474656346303369.boopatt.jobindex"

mlflow.models.set_model(AGENT)

with mlflow.start_run(run_name="joblistings_demo") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            f"langchain=={get_distribution('langchain').version}",
            f"databricks-vectorsearch=={get_distribution('databricks-vectorsearch').version}",
            f"databricks_langchain=={get_distribution('databricks_langchain').version}",
            f"mlflow=={get_distribution('mlflow').version}"
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=vs_index_table_name)
        ]
    )

model_uri = f"runs:/{run.info.run_id}/{model_info.name}"
model_version = mlflow.register_model(model_uri, model_name)
print(f"Model registered: {model_name} | Version: {model_version.version}")

## Step 5 — Deploy with the Databricks Agent Framework

Deploy the registered model using `agents.deploy()`. This does two things automatically:
1. **Creates a Model Serving endpoint** — a scalable REST API for the chatbot
2. **Launches a Review App** — a built-in UI for testing and gathering human feedback on the model's responses

`scale_to_zero=True` keeps costs low by shutting the endpoint down when idle.

Deployment typically takes **15–20 minutes**. The cell will print a dot every 30 seconds while waiting.

In [ ]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate
from databricks import agents

deployment_info = agents.deploy(
    model_name,
    model_version=model_version.version,
    scale_to_zero=True,
    environment_vars={"VS_INDEX_NAME": vs_index_table_name}
)

w = WorkspaceClient()
print(f"Deploying endpoint: {deployment_info.endpoint_name}")
print("Waiting for endpoint to be ready (15–20 min)...", end="", flush=True)

while (
    w.serving_endpoints.get(deployment_info.endpoint_name).state.ready == EndpointStateReady.NOT_READY
    or w.serving_endpoints.get(deployment_info.endpoint_name).state.config_update == EndpointStateConfigUpdate.IN_PROGRESS
):
    print(".", end="", flush=True)
    time.sleep(30)

print(f"\nEndpoint is ready: {deployment_info.endpoint_name}")
print(f"Review App URL: {deployment_info.review_app_url}")